## Executive Summary

**Purpose:** Turn the designed gears and parts into what a gear shop and a machine shop need: the gear data table for each drawing, tooth thickness limits as measurable span and over-ball dimensions, the accuracy class with its tolerance values, fits as numeric limits, heat treatment and case depth, and a tooth outline for prototypes.

**What it does:**

- `iso286_limits` gives the limits of a fit such as H7, G6, j5, n6 for a nominal size (ISO 286-2 tables).
- `iso1328_tolerances` computes the single pitch, cumulative pitch, profile, helix and runout tolerances of ISO 1328-1:2013 for a flank tolerance class.
- `backlash_allowances` splits the minimum backlash of both planetary meshes into tooth thickness allowances for sun, planet and ring.
- `gear_data_sheet` assembles the ISO 1340 / DIN 3966-1 data table of one gear, with span or over-ball limits computed from the thickness allowances.
- `gear_outline` and `write_dxf` give the closed tooth outline of a gear as points or as a DXF polyline, for wire EDM or laser-cut prototypes.

## How It Fits Into the Gear Design Workflow

**Upstream:** `gears.involute` (geometry and measurement formulas), `gears.layout` (the set), `gears.rating` (materials), `gears.components` (fits named on the parts).

**Downstream:** `gears.design.DesignPackage`, which writes the data sheets, the drawing and the report to files.

In [ ]:
#| default_exp gears.manufacturing

In [ ]:
#| export
from __future__ import annotations

import math
from dataclasses import dataclass

import numpy as np

from emachines.gears.involute import GearGeometry, inv, inv_inverse, span_teeth, span_measurement, ball_diameter, ball_measurement
from emachines.gears.layout import PlanetarySet
from emachines.gears.rating import GearMaterial

In [ ]:
#| hide
# drawing helpers for the explanatory figures (not exported)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Polygon, Rectangle
from emachines.gears import layout as _lay, manufacturing as _mfg, involute as _inv   # exported modules, so figures can sit ahead of the definitions

_PS = _lay.PlanetarySet.build(18, 27, 72, m=1.5, n_planets=3, b=20)

INK, BLUE, RED, GREEN, GREY = "#222222", "#1f5fa8", "#b8322a", "#2a8a4a", "#888888"
C_SUN, C_PLANET, C_RING, C_STEEL = "#d8e8d0", "#f6e7c8", "#dfe7f1", "#bdbdbd"

def _rot(P, a):
    c, s_ = np.cos(a), np.sin(a)
    return np.asarray(P) @ np.array([[c, s_], [-s_, c]])

def _label(ax, text, xy, xytext, color=INK, ha="left", fs=9):
    ax.annotate(text, xy=xy, xytext=xytext, fontsize=fs, color=color, ha=ha, va="center",
                arrowprops=dict(arrowstyle="-", color=color, lw=0.7, shrinkA=0, shrinkB=0))

def _dim(ax, p0, p1, text="", off=(0, 0), color=BLUE, fs=9, ha="center"):
    ax.annotate("", xy=p1, xytext=p0, arrowprops=dict(arrowstyle="<->", color=color, lw=0.8, shrinkA=0, shrinkB=0))
    if text:
        ax.text((p0[0] + p1[0]) / 2 + off[0], (p0[1] + p1[1]) / 2 + off[1], text, color=color, fontsize=fs,
                ha=ha, va="center", bbox=dict(fc="white", ec="none", pad=0.6, alpha=0.85))

def _arc(ax, r, a0, a1, color=BLUE, ls="--", lw=0.7):
    t = np.linspace(a0, a1, 200)
    ax.plot(r * np.cos(t), r * np.sin(t), color=color, ls=ls, lw=lw)

## ISO 286 Fits: Theory

A fit such as H7 names a *tolerance zone* around the nominal size: the letter gives the **fundamental deviation**, the deviation of the zone edge nearest the zero line, and the number gives the **standard tolerance grade** IT, the width of the zone. A hole letter is upper case (deviations ES upper, EI lower), a shaft letter lower case (es, ei). Both come from the ISO 286-2 tables for the size range that contains the nominal size $d$, $D_{i-1} < d \le D_i$ (here IT5–IT12 and the letters used on these parts, sizes up to 500 mm).

In [ ]:
#| echo: false
def _fits_figure():
    d = 12.0
    holes, shafts = ["H7", "G6", "N7"], ["g6", "h6", "js6", "j5", "k6", "n6", "p6"]
    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.axhline(0, color=INK, lw=1.0)
    ax.text(-0.35, 0, "zero line\n(nominal Ø12)", ha="right", va="center", fontsize=8.5)
    for i, f in enumerate(holes + shafts):
        up, lo = (1000 * v for v in _mfg.iso286_limits(d, f))
        hole = f[0].isupper()
        x = i + (0 if hole else 0.6)
        ax.add_patch(Rectangle((x, lo), 0.6, up - lo, fc=C_RING if hole else C_PLANET, ec=INK, lw=0.7,
                               hatch="///" if hole else None))
        ax.text(x + 0.3, max(up, 0) + 2.5, f, ha="center", fontsize=9, fontweight="bold")
    # name the deviations on H7 and n6
    up, lo = (1000 * v for v in _mfg.iso286_limits(d, "H7"))
    _dim(ax, (-0.12, lo), (-0.12, up), "", color=BLUE)
    ax.text(0.3, up + 8, f"ES = +{up:g}, EI = 0", ha="center", fontsize=8.5, color=BLUE)
    ax.text(-0.18, up / 2, "IT7", ha="right", va="center", fontsize=8.5, color=BLUE)
    i_n = len(holes) + shafts.index("p6")
    up, lo = (1000 * v for v in _mfg.iso286_limits(d, "p6"))
    _dim(ax, (i_n + 1.3, 0), (i_n + 1.3, lo), "", color=RED)
    ax.text(i_n + 1.38, lo / 2, "ei = fundamental\ndeviation", ha="left", va="center", fontsize=8.5, color=RED)
    _dim(ax, (i_n + 1.3, lo), (i_n + 1.3, up), "", color=BLUE)
    ax.text(i_n + 1.38, (lo + up) / 2 + 2, "IT6", ha="left", va="center", fontsize=8.5, color=BLUE)
    ax.text(1.5, -27, "holes (upper case)", ha="center", fontsize=9)
    ax.text(len(holes) + 4, -27, "shafts (lower case)", ha="center", fontsize=9)
    ax.set_xlim(-2.4, len(holes + shafts) + 2.4); ax.set_ylim(-30, 38)
    ax.set_ylabel("deviation from nominal [µm]")
    ax.set_xticks([]); [ax.spines[s].set_visible(False) for s in ("top", "right", "bottom")]
    ax.set_title("ISO 286 tolerance zones at Ø12 mm", fontsize=10)
    plt.tight_layout()
_fits_figure()

From the tabulated fundamental deviation and IT, the other limit follows:

| Class | Upper deviation | Lower deviation |
|---|---|---|
| H (hole basis) | $ES = IT$ | $EI = 0$ |
| h (shaft basis) | $es = 0$ | $ei = -IT$ |
| JS, js (symmetric) | $+IT/2$ | $-IT/2$ |
| G | $ES = EI + IT$ | $EI = -es_g$ (mirror of g) |
| g | $es$ (table, negative) | $ei = es - IT$ |
| j5, j6 | $es$ (table) | $ei$ (table) |
| k, m, n, p | $es = ei + IT$ | $ei$ (table; $ei = 0$ for k above IT7) |
| r, s, u (interference fits) | $es = ei_{max} + IT$ | $ei_{min}$ (table) |
| N7 | $ES$ (table) | $EI = ES - IT7$ |
| P (IT8 and coarser) | $ES = -ei_p$ | $EI = ES - IT$ |

Where ISO 286-2 splits one of the size ranges used here for r, s and u (for example 50–65 and 65–80 mm), the zone is widened to cover both halves: the lower deviation is the smaller tabulated $ei$ and the upper deviation the larger $ei$ plus IT. That gives a slightly low minimum interference and a slightly high maximum interference, the safe side for both press-fit checks.

`iso286_limits(d, "H7")` returns the upper and lower deviation in mm; `fmt_fit` writes the drawing text $\varnothing d\ \text{fit}\ (\text{upper}/\text{lower})$.

In [ ]:
#| export
_RANGES = [3, 6, 10, 18, 30, 50, 80, 120, 180, 250, 315, 400, 500]
_IT = {5: [4, 5, 6, 8, 9, 11, 13, 15, 18, 20, 23, 25, 27],
       6: [6, 8, 9, 11, 13, 16, 19, 22, 25, 29, 32, 36, 40],
       7: [10, 12, 15, 18, 21, 25, 30, 35, 40, 46, 52, 57, 63],
       8: [14, 18, 22, 27, 33, 39, 46, 54, 63, 72, 81, 89, 97],
       9: [25, 30, 36, 43, 52, 62, 74, 87, 100, 115, 130, 140, 155],
       10: [40, 48, 58, 70, 84, 100, 120, 140, 160, 185, 210, 230, 250],
       11: [60, 75, 90, 110, 130, 160, 190, 220, 250, 290, 320, 360, 400],
       12: [100, 120, 150, 180, 210, 250, 300, 350, 400, 460, 520, 570, 630]}
# fundamental deviations [µm]: g → es (negative), G → EI; k, m, n, p → ei
_FD = {"g": [-2, -4, -5, -6, -7, -9, -10, -12, -14, -15, -17, -18, -20],
       "k": [0, 1, 1, 1, 2, 2, 2, 3, 3, 4, 4, 4, 5],
       "m": [2, 4, 6, 7, 8, 9, 11, 13, 15, 17, 20, 21, 23],
       "n": [4, 8, 10, 12, 15, 17, 20, 23, 27, 31, 34, 37, 40],
       "p": [6, 12, 15, 18, 22, 26, 32, 37, 43, 50, 56, 62, 68]}
# r, s, u shafts (ei) [µm] as (smaller, larger) value where ISO 286-2 splits a size range here, e.g. 50–65 / 65–80
_FD_RSU = {"r": [(10, 10), (15, 15), (19, 19), (23, 23), (28, 28), (34, 34), (41, 43), (51, 54), (63, 68), (77, 84), (94, 98), (108, 114), (126, 132)],
           "s": [(14, 14), (19, 19), (23, 23), (28, 28), (35, 35), (43, 43), (53, 59), (71, 79), (92, 108), (122, 140), (158, 170), (190, 208), (232, 252)],
           "u": [(18, 18), (23, 23), (28, 28), (33, 33), (41, 48), (60, 70), (87, 102), (124, 144), (170, 210), (236, 284), (315, 350), (390, 435), (490, 540)]}
# N7 holes (ES) [µm]; EI = ES − IT7
_N7_ES = [-4, -4, -4, -5, -7, -8, -9, -10, -12, -14, -14, -16, -17]
# j5, j6 (es, ei) [µm]
_J = {5: [(2, -2), (3, -2), (4, -2), (5, -3), (5, -4), (6, -5), (6, -7), (6, -9), (7, -11), (7, -13), (7, -16), (7, -18), (7, -20)],
      6: [(4, -2), (6, -2), (7, -2), (8, -3), (9, -4), (11, -5), (12, -7), (13, -9), (14, -11), (16, -13), (16, -16), (18, -18), (20, -20)]}


def _idx(d: float) -> int:
    for i, hi in enumerate(_RANGES):
        if d <= hi:
            return i
    raise ValueError("ISO 286 tables here cover sizes up to 500 mm")


def it_grade(d: float, grade: int) -> float:
    "Standard tolerance IT`grade` [mm] for nominal size d [mm]."
    return _IT[grade][_idx(d)] / 1000


def iso286_limits(d: float, fit: str) -> tuple[float, float]:
    """
    (upper, lower) deviation [mm] of the tolerance class `fit` (e.g. 'H7', 'G6', 'N7', 'h11', 'js6', 'j5', 'k6', 'n6', 'p6', 's6', 'u6', 'P9', 'JS9').
    """
    letter, grade = fit.rstrip("0123456789"), int(fit[len(fit.rstrip("0123456789")):])
    i = _idx(d)
    IT = _IT[grade][i] / 1000
    if letter == "H": return IT, 0.0
    if letter == "h": return 0.0, -IT
    if letter in ("js", "JS"): return IT / 2, -IT / 2
    if letter == "G": return -_FD["g"][i] / 1000 + IT, -_FD["g"][i] / 1000
    if letter == "g": return _FD["g"][i] / 1000, _FD["g"][i] / 1000 - IT
    if letter == "j":
        es_um, ei_um = _J[grade][i]
        return es_um / 1000, ei_um / 1000
    if letter in ("k", "m", "n", "p"):
        lower = 0.0 if (letter == "k" and grade > 7) else _FD[letter][i] / 1000
        return lower + IT, lower
    if letter in _FD_RSU:
        lo_um, hi_um = _FD_RSU[letter][i]
        return hi_um / 1000 + IT, lo_um / 1000
    if letter == "N" and grade == 7:
        ES = _N7_ES[i] / 1000
        return ES, ES - IT
    if letter == "P" and grade >= 8:
        ES = -_FD["p"][i] / 1000
        return ES, ES - IT
    raise ValueError(f"tolerance class {fit} not in this table")


def fmt_fit(d: float, fit: str, digits: int = 3) -> str:
    "Drawing text such as 'Ø12 H7 (+0.018/0)'."
    up, lo = iso286_limits(d, fit)
    f = lambda v: "0" if abs(v) < 5e-7 else f"{v:+.{digits}f}"
    return f"Ø{d:g} {fit} ({f(up)}/{f(lo)})"

In [ ]:
print(fmt_fit(12, "H7"), fmt_fit(12, "n6"), fmt_fit(9, "G6"), fmt_fit(6, "j5"), fmt_fit(29.4, "h11"))
assert iso286_limits(12, "H7") == (0.018, 0.0) and iso286_limits(12, "n6") == (0.023, 0.012)
assert iso286_limits(25, "G6") == (0.020, 0.007) and iso286_limits(40, "P9") == (-0.026, -0.088)

## ISO 1328-1:2013 Flank Tolerances: Theory

The flank tolerances limit how far the real teeth may depart from perfect involutes at perfect spacing. Each is checked on a gear measuring machine:

- **Pitch.** $f_{pi}$ is the error in the spacing of tooth $i$ from its neighbour, $F_{pi} = \sum f_{pi}$ the running sum around the gear; the single pitch tolerance $f_{pT}$ limits $|f_{pi}|$ and the cumulative tolerance $F_{pT}$ limits the spread $F_p = \max F_{pi} - \min F_{pi}$.
- **Profile** (along the roll length $L_\alpha$ from root to tip) and **helix** (across the face width): the total deviation $F_\alpha$, $F_\beta$ is the band containing the whole trace; the form deviation $f_{f\alpha}$, $f_{f\beta}$ is the band around the least-squares mean line; the slope deviation $f_{H\alpha}$, $f_{H\beta}$ is the rise of the mean line over the evaluation range.
- **Runout** $F_r$: the spread of the radial position of a probe placed in each tooth space in turn.

In [ ]:
#| echo: false
def _iso1328_figure():
    g = _PS.sun
    t = _mfg.iso1328_tolerances(g, 6)
    rng = np.random.default_rng(3)
    fig, axs = plt.subplots(1, 3, figsize=(13, 4.3))
    # (a) pitch: single pitch deviations and their running sum
    ax = axs[0]
    k = np.arange(1, g.z + 1)
    fp = 2.5 * np.sin(2 * np.pi * k / g.z) + rng.normal(0, 2.0, g.z)
    fp -= fp.mean()
    Fp = np.cumsum(fp)
    ax.bar(k, fp, color=C_PLANET, ec=INK, lw=0.5, label="single pitch deviation $f_{pi}$")
    ax.plot(k, Fp, "-o", ms=3, color=BLUE, lw=1, label="cumulative $F_{pi} = \\sum f_{pi}$")
    ax.axhspan(-t["f_pT"], t["f_pT"], color=GREEN, alpha=0.08)
    ax.axhline(t["f_pT"], color=GREEN, lw=0.7, ls="--"); ax.axhline(-t["f_pT"], color=GREEN, lw=0.7, ls="--")
    ax.text(g.z + 0.5, t["f_pT"] + 0.6, f"$\\pm f_{{pT}}$ = {t['f_pT']:g}", fontsize=8.5, color=GREEN, ha="right")
    i0, i1 = int(Fp.argmin()), int(Fp.argmax())
    xd = g.z + 1.6
    ax.hlines([Fp.min(), Fp.max()], [k[i0], k[i1]], xd, color=BLUE, lw=0.5, ls=":")
    _dim(ax, (xd, Fp.min()), (xd, Fp.max()), "", color=BLUE)
    ax.text(xd + 0.4, Fp.min() + 0.75 * (Fp.max() - Fp.min()), "$F_p$\n(≤ $F_{pT}$)", fontsize=8.5, color=BLUE, va="center")
    ax.set_xlim(0, g.z + 5); ax.set_xlabel("tooth number"); ax.set_ylabel("deviation [µm]")
    ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.17), ncol=1, frameon=False)
    ax.set_title("(a) pitch", fontsize=10)
    # (b, c) profile and helix charts
    for ax, name, L, tot, form, slope in ((axs[1], "profile", "roll length $L_\\alpha$", "F_\\alpha", "f_{f\\alpha}", "f_{H\\alpha}"),
                                          (axs[2], "helix", "face width $L_\\beta$", "F_\\beta", "f_{f\\beta}", "f_{H\\beta}")):
        s_ = np.linspace(0, 1, 300)
        dev = 3.0 * s_ + 1.2 * np.sin(9 * s_) + 0.6 * np.sin(23 * s_ + 1)
        c = np.polyfit(s_, dev, 1); mean = np.polyval(c, s_)
        ax.plot(dev, s_, color=INK, lw=1.0, label="measured trace")
        ax.plot(mean, s_, color=RED, lw=0.9, ls="--", label="mean line (least squares)")
        r = dev - mean
        ax.plot(mean + r.max(), s_, color=RED, lw=0.5, ls=":"); ax.plot(mean + r.min(), s_, color=RED, lw=0.5, ls=":")
        y = 1.07
        _dim(ax, (dev.min(), y), (dev.max(), y), "", color=BLUE)
        ax.text((dev.min() + dev.max()) / 2, y + 0.04, f"${tot}$ (total)", ha="center", fontsize=8.5, color=BLUE)
        _dim(ax, (mean[0], -0.07), (mean[-1], -0.07), "", color=RED)
        ax.text((mean[0] + mean[-1]) / 2, -0.13, f"${slope}$ (slope)", ha="center", fontsize=8.5, color=RED)
        yy = 0.55; i = int(yy * 299)
        _dim(ax, (mean[i] + r.min(), yy), (mean[i] + r.max(), yy), "", color=GREEN)
        ax.text(mean[i] + r.max() + 0.2, yy, f"${form}$ (form)", fontsize=8.5, color=GREEN, va="center")
        ax.set_ylim(-0.2, 1.17); ax.set_xlim(dev.min() - 1.2, dev.max() + 1.8)
        ax.set_ylabel(L); ax.set_xlabel("deviation [µm]"); ax.set_yticks([])
        ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.17), ncol=1, frameon=False)
        ax.set_title(f"({'b' if name == 'profile' else 'c'}) {name} chart", fontsize=10)
    plt.tight_layout()
_iso1328_figure()

The 2013 edition defines the tolerances of flank tolerance class $A$ (1 to 11; the old grades 5–8 map roughly to classes 5–8) by formulas in the reference diameter $d$, module $m$ and face width $b$ (mm, results in µm), each scaled by $(\sqrt2)^{A-5}$:

| Tolerance | Formula at class 5 |
|---|---|
| single pitch $f_{pT}$ | $0.001d + 0.4m + 5$ |
| cumulative pitch (total) $F_{pT}$ | $0.002d + 0.55\sqrt d + 0.7m + 12$ |
| total profile $F_{\alpha T}$ | $3.2\sqrt m + 0.22\sqrt d + 0.7$ |
| profile form $f_{f\alpha T}$ | $2.5\sqrt m + 0.17\sqrt d + 0.5$ |
| profile slope $f_{H\alpha T}$ | $2\sqrt m + 0.14\sqrt d + 0.5$ |
| total helix $F_{\beta T}$ | $0.07\sqrt d + 0.45\sqrt b + 4$ |
| helix form and slope $f_{f\beta T} = f_{H\beta T}$ | $0.05\sqrt d + 0.35\sqrt b + 4$ |
| radial runout $F_{rT}$ | $0.9\,F_{pT}$ (ISO 1328-2 guidance) |

A tolerance of class $A$ is the class-5 value times the step factor:

$$T_A = T_5\,(\sqrt2)^{A-5},$$

so each class is $\sqrt2$ coarser than the one before and two classes double the tolerance. Values above 10 µm are rounded to whole µm, values from 5 to 10 µm to 0.5 µm, and smaller values to 0.1 µm.

In [ ]:
#| export
def _round_tol(v: float) -> float:
    if v > 10: return float(round(v))
    if v >= 5: return round(v * 2) / 2
    return round(v, 1)


def iso1328_tolerances(g: GearGeometry, A: int) -> dict[str, float]:
    "ISO 1328-1:2013 flank tolerances [µm] of gear g for flank tolerance class A."
    d, m, b = g.d, g.m, g.b
    k = math.sqrt(2) ** (A - 5)
    F_p = (0.002 * d + 0.55 * math.sqrt(d) + 0.7 * m + 12) * k
    return {"f_pT": _round_tol((0.001 * d + 0.4 * m + 5) * k),
            "F_pT": _round_tol(F_p),
            "F_alphaT": _round_tol((3.2 * math.sqrt(m) + 0.22 * math.sqrt(d) + 0.7) * k),
            "f_falphaT": _round_tol((2.5 * math.sqrt(m) + 0.17 * math.sqrt(d) + 0.5) * k),
            "f_HalphaT": _round_tol((2.0 * math.sqrt(m) + 0.14 * math.sqrt(d) + 0.5) * k),
            "F_betaT": _round_tol((0.07 * math.sqrt(d) + 0.45 * math.sqrt(b) + 4) * k),
            "f_HbetaT": _round_tol((0.05 * math.sqrt(d) + 0.35 * math.sqrt(b) + 4) * k),
            "F_rT": _round_tol(0.9 * F_p)}

In [ ]:
t6 = iso1328_tolerances(GearGeometry(z=18, m=1.5, b=20), 6)
print(t6)
assert t6["F_pT"] > t6["f_pT"] and iso1328_tolerances(GearGeometry(z=18, m=1.5, b=20), 7)["F_pT"] > t6["F_pT"]

## Backlash and Tooth Thickness Allowances: Theory

Gears are made with their teeth slightly thinner than the theoretical value so that the mesh has backlash: with the drive flanks in contact, the opposite flanks stand apart by the normal backlash $j_{bn}$, measured along their line of action. The thinning of each tooth is specified by two *thickness allowances* on the reference circle, the upper $A_{sne}$ and the lower $A_{sni}$:

In [ ]:
#| echo: false
def _backlash_figure():
    ps = _PS
    sun, pl = ps.sun, ps.planet
    fig, axs = plt.subplots(1, 2, figsize=(12, 5.6))
    # (a) thickness allowances on one sun tooth (allowances exaggerated ×8)
    ax = axs[0]
    k = 8
    al = _mfg.backlash_allowances(ps, 6)["allowances"]["sun"]
    for ds, st, lab in ((0.0, "--", "nominal, $s$"), (k * al.A_sne, "-", "upper limit, $s + A_{sne}$"),
                        (k * al.A_sni, ":", "lower limit, $s + A_{sni}$")):
        P = _rot(_mfg.gear_outline(sun, ds), np.pi / 2)
        P = np.vstack([P, P[:1]])
        ax.plot(*P.T, color=INK if ds else GREY, ls=st, lw=1.0, label=lab)
    r = sun.d / 2
    _arc(ax, r, np.pi / 2 - 0.35, np.pi / 2 + 0.35, color=BLUE, ls="-.", lw=0.7)
    ax.text(-2.5, r - 0.25, "reference\ncircle $d$", color=BLUE, fontsize=8.5, ha="left", va="top")
    for ds, yo, txt in ((0.0, 2.3, "$s$"), (k * al.A_sne, 1.55, "$s + A_{sne}$"), (k * al.A_sni, 0.8, "$s + A_{sni}$")):
        h = (sun.s + ds) / sun.d
        y = r + yo
        xl, xr = -r * np.sin(h), r * np.sin(h)
        ax.plot([xl, xl], [r * np.cos(h), y], color=BLUE, lw=0.4); ax.plot([xr, xr], [r * np.cos(h), y], color=BLUE, lw=0.4)
        ax.plot([xr, 1.7], [y, y], color=BLUE, lw=0.4, ls=":")
        _dim(ax, (xl, y), (xr, y), "", color=BLUE)
        ax.text(1.75, y, txt, color=BLUE, fontsize=9, va="center")
    _label(ax, f"$T_{{sn}} = A_{{sne}} - A_{{sni}}$\n(tolerance band)", (-r * np.sin((sun.s + k * (al.A_sne + al.A_sni) / 2) / sun.d) - 0.03, r - 0.1),
           (-2.5, r + 1.3), color=RED)
    ax.set_xlim(-2.6, 3.9); ax.set_ylim(r - 3.4, sun.da / 2 + 0.6); ax.set_aspect("equal"); ax.axis("off")
    ax.legend(fontsize=8.5, loc="lower left", frameon=False)
    ax.set_title(f"(a) thickness allowances on the sun (exaggerated ×{k})", fontsize=10)
    # (b) backlash in the sun–planet mesh, drive flanks in contact
    ax = axs[1]
    ds = -0.25
    aw = ps.sun_planet.alpha_w
    jn = -2 * ds * np.cos(sun.alpha)
    rw2 = pl.db / 2 / np.cos(aw)
    psi = np.pi - np.pi / pl.z
    delta = 0.5 * (jn / np.cos(aw)) / rw2          # from the centred position, turn the planet by half the backlash
    Ps = _rot(_mfg.gear_outline(sun, ds), np.pi / 2)
    Pp = _rot(_rot(_mfg.gear_outline(pl, ds), psi + delta), np.pi / 2) + np.array([0, ps.a_w])
    ax.add_patch(Polygon(Ps, closed=True, fc=C_SUN, ec=INK, lw=0.8))
    ax.add_patch(Polygon(Pp, closed=True, fc=C_PLANET, ec=INK, lw=0.8))
    rw1 = sun.db / 2 / np.cos(aw)
    C = np.array([0, rw1])
    for sgn, col, lab in ((-1, GREEN, "line of action of the drive flanks (in contact)"),
                          (1, RED, "line of action of the opposite flanks")):
        u = np.array([np.cos(aw), sgn * np.sin(aw)])
        ax.plot(*np.array([C - 3.4 * u, C + 3.4 * u]).T, color=col, lw=0.8, ls="--", label=lab)
    ax.plot(*C, "o", color=INK, ms=3)
    ax.text(C[0] - 0.1, C[1] - 0.35, "C", fontsize=9, ha="right")
    # the gap between the opposite flanks, along their line of action
    from matplotlib.path import Path
    u = np.array([np.cos(aw), np.sin(aw)])
    t = np.linspace(-3, 3, 6001)
    L = C + t[:, None] * u
    inS, inP = Path(Ps).contains_points(L), Path(Pp).contains_points(L)
    last_S = np.flatnonzero(inS)[-1]                # leave the sun tooth along +u ...
    first_P = last_S + 1 + np.flatnonzero(inP[last_S + 1:])[0]   # ... and cross the gap to the planet
    run = [last_S + 1, first_P - 1]
    p0, p1 = L[run[0]], L[run[-1]]
    _dim(ax, p0, p1, "", color=RED)
    _label(ax, f"$j_{{bn}}$ = {np.linalg.norm(p1 - p0):.2f} mm (normal backlash),\nmeasured along the line of action", (p1 + p0) / 2, (1.0, rw1 + 2.7), color=RED)
    ax.set_xlim(-4.2, 4.2); ax.set_ylim(rw1 - 3.2, rw1 + 3.4); ax.set_aspect("equal"); ax.axis("off")
    ax.legend(fontsize=8.5, loc="lower center", frameon=False, ncol=1)
    ax.set_title(f"(b) sun–planet mesh, drive flanks touching (thinning exaggerated)", fontsize=10)
    plt.tight_layout()
_backlash_figure()

A thinning $\Delta s$ of the arc tooth thickness on the reference circle opens the mesh by $-\Delta s\cos\alpha$ along the line of action, so two gears with allowances $A_1$, $A_2$ have $j_{bn} = -(A_1 + A_2)\cos\alpha$ at the nominal centre distance. The smallest normal backlash a mesh should have (ISO/TR 10064-2, also AGMA 2002) is

$$j_{bn,min} = \tfrac{2}{3}\,(0.06 + 0.0005\,a_w + 0.03\,m)\ \text{mm}.$$

A centre distance error $\Delta a$ changes the backlash by $2\Delta a\sin\alpha_w$: a shorter distance takes backlash away on an external mesh, a longer one on an internal mesh. With the centre distance held to $\pm f_a$, the carrier pin-hole position tolerance,

$$f_a = \tfrac12\,IT7(a_w),$$

the thickness reductions of the two gears at the reference circle must add up to at least

$$S = \frac{j_{bn,min} + 2f_a\sin\alpha_w}{\cos\alpha}.$$

This is evaluated for both meshes, $S_{sp}$ (sun–planet, $\alpha_w$ of that mesh) and $S_{pr}$ (planet–ring). The planet is in both meshes, so it takes half of the larger requirement and the sun and ring make up the rest of their own mesh, but never less than a quarter of it. With $\lceil\ \rceil_{5}$ meaning "rounded up to 5 µm", the upper allowances (negative: tooth thinner than nominal) are

$$A_{sne,p} = -\Big\lceil \tfrac12 \max(S_{sp}, S_{pr}) \Big\rceil_5,\qquad A_{sne,s} = -\Big\lceil \max\big(S_{sp} + A_{sne,p},\ 0.25\,S_{sp}\big)\Big\rceil_5,\qquad A_{sne,r} = -\Big\lceil \max\big(S_{pr} + A_{sne,p},\ 0.25\,S_{pr}\big)\Big\rceil_5.$$

The *lower* allowance is $A_{sni} = A_{sne} - T_{sn}$. The tolerance $T_{sn}$ must at least cover the thickness change caused by runout (DIN 3967 practice); 15 µm is added for the measuring and machining spread:

$$T_{sn} = \big\lceil 2F_{rT}\tan\alpha + 0.015\ \text{mm} \big\rceil_5 .$$

Agree $T_{sn}$ with your gear shop. A shop that grinds can often hold less, a shop that shapes the ring may need more.

The backlash range of each mesh then follows from the allowance limits and the two ends of the centre distance tolerance:

$$j_{bn,min} = -(A_{sne,1} + A_{sne,2})\cos\alpha - 2f_a\sin\alpha_w,\qquad j_{bn,max} = -(A_{sni,1} + A_{sni,2})\cos\alpha + 2f_a\sin\alpha_w .$$

The smallest must reach the required value; the largest is reported so you can check it against the positioning accuracy your application needs.

In [ ]:
#| export
@dataclass(frozen=True)
class ThicknessAllowance:
    "Tooth thickness allowances of one gear on the reference circle [mm] (negative = thinner)."
    A_sne: float
    A_sni: float

    @property
    def T_sn(self) -> float: return self.A_sne - self.A_sni


def _round_up(v: float, step: float = 0.005) -> float:
    return math.ceil(v / step - 1e-9) * step


def backlash_allowances(ps: PlanetarySet, A: int) -> dict:
    """
    Thickness allowances of sun, planet and ring, plus min/max backlash of both meshes [mm].

    A: ISO 1328-1 flank tolerance class (sets the runout used for the thickness tolerance).
    """
    a_w = ps.a_w
    f_a = it_grade(a_w, 7) / 2
    m = ps.sun.m
    jmin = 2 / 3 * (0.06 + 0.0005 * a_w + 0.03 * m)
    ca = math.cos(ps.sun.alpha)
    S_sp = (jmin + 2 * f_a * math.sin(ps.sun_planet.alpha_w)) / ca
    S_pr = (jmin + 2 * f_a * math.sin(ps.planet_ring.alpha_w)) / ca
    planet_up = _round_up(max(S_sp, S_pr) / 2)
    ups = {"sun": _round_up(max(S_sp - planet_up, 0.25 * S_sp)), "planet": planet_up,
           "ring": _round_up(max(S_pr - planet_up, 0.25 * S_pr))}
    out = {}
    for key, g in (("sun", ps.sun), ("planet", ps.planet), ("ring", ps.ring)):
        Fr = iso1328_tolerances(g, A)["F_rT"] / 1000
        T = _round_up(2 * Fr * math.tan(g.alpha) + 0.015)
        out[key] = ThicknessAllowance(-ups[key], -ups[key] - T)
    j = {}
    for mesh, k1, k2, aw in (("sun–planet", "sun", "planet", ps.sun_planet.alpha_w),
                             ("planet–ring", "planet", "ring", ps.planet_ring.alpha_w)):
        jn_min = -(out[k1].A_sne + out[k2].A_sne) * ca - 2 * f_a * math.sin(aw)
        jn_max = -(out[k1].A_sni + out[k2].A_sni) * ca + 2 * f_a * math.sin(aw)
        j[mesh] = (jn_min, jn_max)
    return {"allowances": out, "backlash_n": j, "j_bn_min_required": jmin, "f_a": f_a}

## Gear Data Sheet: Theory

ISO 1340 and DIN 3966-1 list what a gear drawing must carry so that the gear can be cut and inspected without the designer's calculation: the basic geometry ($z$, $m$, $\alpha$, $x$ and the basic rack), the diameters, the thickness with its allowances and the matching measurement limits, the flank tolerance class with its values, the mating gear and centre distance, and the material, heat treatment and surface data. `gear_data_sheet` produces those rows. On one tooth:

In [ ]:
#| echo: false
def _datasheet_tooth_figure():
    ps = _PS
    g = ps.sun
    d_Nf = ps.sun_planet.d_Nf1
    chd = max(0.3, 0.175 * g.m)
    ra, rf = g.da / 2, g.df / 2
    P = _rot(_mfg.gear_outline(g, 0.0, n_flank=60), np.pi / 2)
    fig, ax = plt.subplots(figsize=(10, 6.2))
    body = Polygon(P, closed=True, fc=C_SUN, ec="none")
    ax.add_patch(body)
    a0, a1 = np.pi / 2 - 0.42, np.pi / 2 + 0.42
    for dd, col, ls in ((g.da, BLUE, "--"), (d_Nf, RED, "-"), (g.d_form, GREEN, "--"), (g.df, BLUE, "-.")):
        _arc(ax, dd / 2, a0, a1, color=col, ls=ls, lw=0.8)
    ax.set_xlim(-5.2, 5.2); ax.set_ylim(rf - 1.7, ra + 1.5); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title("What the data sheet fixes on one tooth (sun, m = 1.5)", fontsize=10)
    plt.tight_layout()
    # hardened case: a stroke of width 2·CHD along the outline, clipped to the gear body, leaves a band CHD deep
    fig.canvas.draw()
    px = ax.transData.transform([[1, 0]])[0, 0] - ax.transData.transform([[0, 0]])[0, 0]
    case, = ax.plot(*np.vstack([P, P[:1]]).T, color="#9fb7d6", lw=2 * chd * px * 72 / fig.dpi, solid_joinstyle="round")
    case.set_clip_path(body)
    ax.plot(*np.vstack([P, P[:1]]).T, color=INK, lw=0.9)
    tip = P[np.abs(np.hypot(*P.T) - ra) < 1e-6]
    xt = tip[np.abs(tip[:, 0]) < 2, 0].max()
    _label(ax, "$d_a$ tip diameter (h11)", (-0.9, np.sqrt(ra**2 - 0.81)), (-5.1, ra + 0.8), color=BLUE)
    _label(ax, f"tip edge chamfer or round\n0.1–0.15 m = {0.1 * g.m:.2f}–{0.15 * g.m:.2f} mm", (xt, np.sqrt(ra**2 - xt**2)), (1.3, ra + 0.9))
    x = 2.75
    _label(ax, "$d_{Nf}$ start of active\nprofile (from the mesh)", (x, np.sqrt((d_Nf / 2)**2 - x**2)), (3.3, d_Nf / 2 + 0.9), color=RED)
    x = 3.1
    _label(ax, "$d_{Ff}$ root form circle\n($d_{Ff} \\leq d_{Nf}$)", (x, np.sqrt((g.d_form / 2)**2 - x**2)), (3.3, g.d_form / 2 - 0.9), color=GREEN)
    a = np.pi / 2 + np.pi / g.z
    _label(ax, "$d_f$ root diameter (max)", (rf * np.cos(a), rf * np.sin(a)), (-5.1, rf - 1.0), color=BLUE)
    yl = g.d / 2
    xl = P[(np.abs(P[:, 0]) < 2) & (P[:, 0] < 0) & (np.abs(P[:, 1] - yl) < 0.3), 0].max()
    _label(ax, f"hardened case, CHD 550 HV1\n0.15–0.2 m, ≥ 0.3 mm (≈ {chd:.2f} mm here)", (xl + 0.5 * chd, yl + 0.6), (-5.1, ra - 0.25))
    ax.text(0, yl + 0.3, "core", ha="center", fontsize=9)
    ax.text(3.6, rf - 1.4, "flanks Ra ≤ 0.4 µm (ground)\nroot Ra ≤ 3.2 µm, no grinding notch", fontsize=8.5, ha="center")
_datasheet_tooth_figure()

**Inspection limits.** The span $W_k$ (external gears) and the dimension over balls $M_d$ are evaluated at the two thickness limits

$$s_{max} = s + A_{sne},\qquad s_{min} = s + A_{sni}$$

with the formulas of `gears.involute`, giving $W_k(s_{max}) / W_k(s_{min})$ and $M_d(s_{max}) / M_d(s_{min})$. They are the dimensions the gear inspector measures. On an internal gear a thinner tooth means a wider space and a *larger* dimension between balls, so its upper limit is $M_d(s_{min})$.

**Root form diameter.** The drawing requires the involute to reach the root form diameter $d_{Ff}$, which must lie below the start of the active profile $d_{Nf}$ found by `gears.involute.mesh_check` (above it for an internal gear): $d_{Ff} \le d_{Nf}$ (external), $d_{Ff} \ge d_{Nf}$ (internal).

**Tip diameter and centre distance tolerances.** External gears h11, internal gears H11 (the tip diameter only needs to be held loosely, because it only affects the contact ratio and the clearances checked in `gears.layout`). The centre distance is given as $a_w \pm f_a$ with $f_a = IT7(a_w)/2$, as used for the backlash.

**Case depth.** For carburized gears the effective case depth (CHD to 550 HV, ISO 2639) after grinding is chosen in proportion to the module:

$$CHD = \max(0.3,\ 0.15\,m)\ \text{to}\ \max(0.45,\ 0.2\,m)\ \text{mm}.$$

That depth is deep enough to carry the subsurface shear stress under the contact, and shallow enough that the tooth tip does not through-harden. Specify the depth before grinding as this plus the grinding stock (about 0.05–0.1 mm per flank). For nitrided gears, the nitriding hardness depth (NHD, core + 50 HV) is 0.3–0.5 mm with a compound layer of at most 10 µm.

**Surfaces and edges.** The tip edges get a chamfer or round of $0.1\,m$ to $0.15\,m$. Roughness by finish:

| Finish | Flanks | Root |
|---|---|---|
| ground | Ra ≤ 0.4 µm (Rz ≤ 3) | Ra ≤ 3.2 µm, no grinding notch |
| hobbed or shaped | Ra ≤ 1.6 µm | Ra ≤ 3.2 µm |

**Process route.** Turn the blank with bore and faces as datum, then cut (hob an external gear; shape or power-skive an internal one), then:

| Material | Route after cutting |
|---|---|
| case carburized | carburize, harden, temper → grind bore and faces → grind flanks (or hone or lap if not ground) |
| nitrided | gas nitride after finish cutting (no grinding after) |
| through hardened | grind flanks if the finish is "ground" |

In [ ]:
#| export
def case_depth(g: GearGeometry, mat: GearMaterial) -> str:
    "Recommended hardening depth text for the drawing."
    if mat.kind == "case":
        lo, hi = max(0.3, 0.15 * g.m), max(0.45, 0.2 * g.m)
        return f"CHD 550 HV1 = {lo:.2f}–{hi:.2f} mm after grinding (ISO 2639)"
    if mat.kind == "nitrided":
        return "NHD (core + 50 HV) = 0.30–0.50 mm, compound layer ≤ 10 µm"
    return "through hardened — no case depth"


@dataclass(frozen=True)
class DataRow:
    "One row of a gear data table."
    item: str
    symbol: str
    value: str


def gear_data_sheet(g: GearGeometry, mate: GearGeometry, a_w: float, allowance: ThicknessAllowance,
                    A: int, mat: GearMaterial, finish: str, n_mate_label: str = "",
                    d_active: float | None = None) -> list[DataRow]:
    """
    ISO 1340 / DIN 3966-1 data table rows for gear g meshing with `mate` at centre distance a_w.

    finish: 'ground', 'hobbed' or 'shaped' (sets roughness text and process route).
    d_active: start of active profile (from `gears.involute.mesh_check`) for the form-diameter requirement.
    """
    deg = math.degrees
    s_up, s_lo = g.s + allowance.A_sne, g.s + allowance.A_sni
    tol = iso1328_tolerances(g, A)
    rows = [
        DataRow("Gear", "", g.name),
        DataRow("Type", "", "spur, internal" if g.internal else "spur, external"),
        DataRow("Number of teeth", "z", f"{g.z}"),
        DataRow("Normal module", "m_n", f"{g.m:g} mm"),
        DataRow("Pressure angle", "α_n", f"{deg(g.alpha):g}°"),
        DataRow("Helix angle", "β", "0°"),
        DataRow("Basic rack profile", "", f"ISO 53 profile A: h_aP = {g.ha:g} m, h_fP = {g.hf:g} m, ρ_fP = {g.rho_f:g} m"),
        DataRow("Profile shift coefficient", "x", f"{g.x:+.4f}"),
        DataRow("Reference diameter", "d", f"{g.d:.4f} mm"),
        DataRow("Base diameter", "d_b", f"{g.db:.4f} mm"),
        DataRow("Tip diameter", "d_a", f"{g.da:.3f} mm " + ("H11 " + _dev(g.da, 'H11') if g.internal else "h11 " + _dev(g.da, 'h11'))),
        DataRow("Root diameter", "d_f", f"{g.df:.3f} mm " + ("(min)" if g.internal else "(max)")),
        DataRow("Root form diameter", "d_Ff", (f"≥ {g.d_form:.3f} mm" if g.internal else f"≤ {g.d_form:.3f} mm")
                + (f" (active profile starts at {d_active:.3f} mm)" if d_active else "")),
        DataRow("Face width", "b", f"{g.b:g} mm ±0.1"),
        DataRow("Tooth thickness on d (arc)", "s", f"{g.s:.4f} mm, allowances {allowance.A_sne:+.3f}/{allowance.A_sni:+.3f} mm"),
    ]
    if not g.internal:
        k = span_teeth(g)
        Wu, Wl = span_measurement(g, s_up, k), span_measurement(g, s_lo, k)
        rows.append(DataRow("Span measurement", f"W_k (k = {k})", f"{Wu:.4f} / {Wl:.4f} mm"))
    D = ball_diameter(g)
    Mu, dc = ball_measurement(g, D, s_up)
    Ml, _ = ball_measurement(g, D, s_lo)
    lab = "Dimension between balls" if g.internal else "Dimension over balls"
    hi, lo = (Ml, Mu) if g.internal else (Mu, Ml)
    rows.append(DataRow(lab, f"M_d (D_M = {D:.2f} mm)", f"{hi:.4f} / {lo:.4f} mm"
                         + (" (odd z: two-ball across one tooth space)" if g.z % 2 else "")))
    rows += [
        DataRow("Flank tolerance class", "", f"ISO 1328-1:2013 class {A}"),
        DataRow("Single pitch tolerance", "f_pT", f"{tol['f_pT']:g} µm"),
        DataRow("Total cumulative pitch tolerance", "F_pT", f"{tol['F_pT']:g} µm"),
        DataRow("Total profile tolerance", "F_αT", f"{tol['F_alphaT']:g} µm"),
        DataRow("Profile slope tolerance", "f_HαT", f"±{tol['f_HalphaT']:g} µm"),
        DataRow("Total helix tolerance", "F_βT", f"{tol['F_betaT']:g} µm"),
        DataRow("Helix slope tolerance", "f_HβT", f"±{tol['f_HbetaT']:g} µm (crown 3–5 µm recommended on planets)" if g.name == "planet" else f"±{tol['f_HbetaT']:g} µm"),
        DataRow("Radial runout tolerance", "F_rT", f"{tol['F_rT']:g} µm"),
        DataRow("Mating gear", "", f"{mate.name}, z = {mate.z}{n_mate_label}"),
        DataRow("Centre distance", "a_w", f"{a_w:.4f} mm ±{it_grade(a_w, 7) / 2 * 1000:.0f} µm"),
        DataRow("Material", "", mat.name),
        DataRow("Heat treatment", "", mat.treatment),
        DataRow("Hardness", "", mat.hardness),
        DataRow("Hardening depth", "", case_depth(g, mat)),
        DataRow("Flank roughness", "", {"ground": "Ra ≤ 0.4 µm (Rz ≤ 3)", "hobbed": "Ra ≤ 1.6 µm", "shaped": "Ra ≤ 1.6 µm"}[finish]),
        DataRow("Root roughness", "", "Ra ≤ 3.2 µm, no grinding notch in the root"),
        DataRow("Tip edge", "", f"chamfer or round 0.1–0.15 m ({0.1 * g.m:.2f}–{0.15 * g.m:.2f} mm)"),
        DataRow("Process route", "", _route(g, mat, finish)),
    ]
    return rows


def _dev(d: float, fit: str) -> str:
    up, lo = iso286_limits(d, fit)
    f = lambda v: "0" if abs(v) < 5e-7 else f"{v:+.3f}"
    return f"({f(up)}/{f(lo)})"


def _route(g: GearGeometry, mat: GearMaterial, finish: str) -> str:
    cut = "shape (pinion-type cutter) or power-skive" if g.internal else "hob"
    if mat.kind == "case":
        ht = "carburize, harden, temper"
    elif mat.kind == "nitrided":
        ht = "gas nitride (after finish cutting)"
    else:
        ht = ""
    steps = ["turn blank (bore and faces as datum)", cut]
    if mat.kind == "nitrided":
        steps += [ht]
    elif ht:
        steps += [ht, "grind bore and faces", "grind flanks" if finish == "ground" else "hone or lap flanks"]
    elif finish == "ground":
        steps += ["grind flanks"]
    return " → ".join(steps)

### Example

Data sheets of the 18/27/72 set, flank class 6:

In [ ]:
from emachines.gears.layout import PlanetarySet
from emachines.gears.rating import MATERIALS
from emachines.gears.involute import mesh_check
ps = PlanetarySet.build(18, 27, 72, m=1.5, n_planets=3, b=20)
bl = backlash_allowances(ps, 6)
print({k: (round(v.A_sne, 3), round(v.A_sni, 3)) for k, v in bl["allowances"].items()},
      {k: tuple(round(x, 3) for x in v) for k, v in bl["backlash_n"].items()}, f"j_bn,min = {bl['j_bn_min_required']:.3f}")
sheet = gear_data_sheet(ps.sun, ps.planet, ps.a_w, bl["allowances"]["sun"], 6, MATERIALS["case_carburized"], "ground",
                        d_active=ps.sun_planet.d_Nf1)
for r in sheet:
    print(f"{r.item:34s} {r.symbol:20s} {r.value}")
for mesh, (jmin, jmax) in bl["backlash_n"].items():
    assert jmin >= bl["j_bn_min_required"] - 1e-9 and jmax > jmin

In [ ]:
ring_sheet = gear_data_sheet(ps.ring, ps.planet, ps.a_w, bl["allowances"]["ring"], 7, MATERIALS["qt_nitrided"], "shaped",
                             d_active=ps.planet_ring.d_Nf2)
for r in ring_sheet[10:17]:
    print(f"{r.item:34s} {r.symbol:20s} {r.value}")

## Tooth Outline for Prototypes: Theory

`gear_outline` returns the closed outline of the whole gear as points. On an external gear the flanks are exact involutes from the tip down to the root form circle, and the root fillet below them is generated the way a hob makes it: as the envelope of the ISO 53 rack's tip radius rolling on the reference circle. On an internal gear the fillet depends on the shaper cutter, so the outline runs straight from the form circle to the root circle. The outline is right for wire EDM or laser-cut prototypes, test fixtures and drawings; production gears are generated by the hob or shaper from the data sheet. A thickness change $\Delta s$ can be applied (for example the mean of the two allowance limits). `write_dxf` writes the outline as an R12 DXF polyline, in mm.

In [ ]:
#| echo: false
def _outline_figure():
    g = _PS.sun
    m, al, r = g.m, g.alpha, g.d / 2
    rho, y_ref = g.rho_f * m, r + g.x * m
    y_tip = y_ref - g.hf * m
    half_ref = np.pi * m / 4
    fig, axs = plt.subplots(1, 2, figsize=(12.5, 5.8))
    # (a) rack generation of the root fillet
    ax = axs[0]
    P = _rot(_mfg.gear_outline(g, n_flank=60), np.pi / 2 + np.pi / g.z)   # a tooth space centred on +y
    win = Rectangle((-3.0, y_tip - 2.0), 6.0, 10.0, transform=ax.transData)
    gear = ax.add_patch(Polygon(P, closed=True, fc=C_SUN, ec=INK, lw=1.1, zorder=3))
    gear.set_clip_path(win)
    # tool tooth in rack coordinates (u along the rolling line, y radial), filling the gear's tooth space
    y_top = y_ref + 1.0 * m
    u_c = half_ref - (y_ref - (y_tip + rho)) * np.tan(al) - rho / np.cos(al)
    tt = np.linspace(-np.pi / 2, -al, 20)
    right = np.c_[u_c + rho * np.cos(tt), y_tip + rho + rho * np.sin(tt)]
    right = np.vstack([right, [half_ref + (y_top - y_ref) * np.tan(al), y_top]])
    left = right[::-1] * [-1, 1]
    T = np.vstack([left, right])
    for k, phi in enumerate(np.linspace(-0.22, 0.22, 9)):
        X = T[:, 0] - r * phi
        xg, yg = X * np.cos(phi) + T[:, 1] * np.sin(phi), -X * np.sin(phi) + T[:, 1] * np.cos(phi)
        ln, = ax.plot(xg, yg, color=RED if phi == 0 else GREY, lw=1.0 if phi == 0 else 0.5, zorder=4 if phi == 0 else 2)
        ln.set_clip_path(win)
    _arc(ax, r, np.pi / 2 - 0.2, np.pi / 2 + 0.2, color=BLUE, ls="-.", lw=0.7)
    ax.plot([-3.0, 3.2], [y_ref, y_ref], color=RED, ls="--", lw=0.7, zorder=5)
    ax.plot([-3.0, 3.2], [y_tip, y_tip], color=RED, ls=":", lw=0.7, zorder=5)
    ax.text(3.25, y_ref, "tool reference line", color=RED, fontsize=8.5, va="center")
    ax.text(3.25, y_tip, "tool tip line = root circle $d_f$", color=RED, fontsize=8.5, va="center")
    _label(ax, "reference circle $d$ (rolls\nwithout slip on the tool reference line)", (1.5, np.sqrt(r**2 - 2.25)), (3.25, r - 0.6), color=BLUE)
    _dim(ax, (-0.6, np.sqrt(r**2 - 0.36)), (-0.6, y_ref), "", color=BLUE)
    ax.text(-0.7, (r + y_ref) / 2, "$x\\,m$", color=BLUE, fontsize=9, ha="right", va="center", zorder=6)
    c = np.array([u_c, y_tip + rho])
    ax.add_patch(Circle(c, rho, fill=False, ec=RED, lw=0.6, ls=":"))
    _label(ax, "tool tip radius $\\rho_{fP} = 0.38\\,m$", c + rho * np.array([np.cos(-1.0), np.sin(-1.0)]), (3.25, y_tip - 0.9), color=RED)
    _label(ax, "fillet = envelope of the\ntool tip positions", (-1.05, y_tip + 0.35), (3.25, y_tip - 1.7), color=INK)
    ax.set_xlim(-3.1, 7.6); ax.set_ylim(y_tip - 1.6, y_top + 0.4); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title("(a) root fillet generated by the ISO 53 rack (hob)", fontsize=10)
    # (b) polar description of the involute flank
    ax = axs[1]
    rb, ra = g.db / 2, g.da / 2
    psi_ref = g.s / g.d
    inv = lambda a: np.tan(a) - a
    rr = np.linspace(rb, ra, 100)
    ar = np.arccos(rb / rr)
    half = psi_ref + inv(al) - inv(ar)
    for sg in (-1, 1):
        ax.plot(rr * np.sin(sg * half), rr * np.cos(sg * half), color=INK, lw=1.1)
    ax.plot([-rr[-1] * np.sin(half[-1]), rr[-1] * np.sin(half[-1])], [ra * np.cos(half[-1])] * 2, color=INK, lw=1.1)
    _arc(ax, rb, np.pi / 2 - 0.5, np.pi / 2 + 0.5, color=GREEN, ls="--")
    ax.text(rb * np.cos(np.pi / 2 - 0.2), rb * np.sin(np.pi / 2 - 0.2) - 0.2, "base circle $r_b$", color=GREEN, fontsize=8.5, va="top")
    ax.plot([0, 0], [rb - 1.2, ra + 0.8], color=GREY, lw=0.6, ls="-.")
    ax.text(0.05, ra + 0.7, "tooth centre line", fontsize=8.5, color=GREY)
    ry = r + 0.8 * m
    i = np.argmin(np.abs(rr - ry))
    py = np.array([-rr[i] * np.sin(half[i]), rr[i] * np.cos(half[i])])
    ax.plot([0, py[0] * 1.0], [0, py[1]], color=BLUE, lw=0.7)
    ax.plot(*py, "o", color=BLUE, ms=3)
    t = np.linspace(np.pi / 2, np.pi / 2 + half[i], 40)
    ax.plot((rb - 0.8) * np.cos(t), (rb - 0.8) * np.sin(t), color=BLUE, lw=0.9)
    ax.text(0.1, rb - 0.75, "$\\psi(r)$", color=BLUE, fontsize=9.5, ha="left", va="center")
    ax.text(py[0] - 0.15, py[1], "point at radius $r$", color=BLUE, fontsize=8.5, ha="right", va="center")
    # involute start on the base circle and the angle inv(α_r)
    th0 = half[0]
    p0 = np.array([-rb * np.sin(th0), rb * np.cos(th0)])
    ax.plot(*p0, "o", color=GREEN, ms=3)
    _label(ax, "involute starts on the base circle", p0, (-4.5, rb - 0.6), color=GREEN)
    t = np.linspace(np.pi / 2 + half[i], np.pi / 2 + th0, 30)
    ax.plot((rb - 1.6) * np.cos(t), (rb - 1.6) * np.sin(t), color=RED, lw=0.9)
    ax.plot([0, p0[0] * 0.98], [0, p0[1] * 0.98], color=RED, lw=0.5, ls=":")
    _label(ax, "$\\mathrm{inv}\\,\\alpha_r = \\tan\\alpha_r - \\alpha_r$,\n$\\cos\\alpha_r = r_b / r$",
           ((rb - 1.6) * np.cos(t[15]), (rb - 1.6) * np.sin(t[15])), (-4.5, rb - 2.3), color=RED)
    ax.set_xlim(-4.8, 3.6); ax.set_ylim(rb - 2.8, ra + 1.2); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title("(b) the flank in polar form: half tooth angle $\\psi(r)$", fontsize=10)
    plt.tight_layout()
_outline_figure()

**Flanks.** A point of the involute at radius $r$ lies at the polar angle $\mathrm{inv}\,\alpha_r$ from where the involute leaves the base circle, with $\cos\alpha_r = r_b/r$. Measured from the tooth centre line, the flank is therefore at the half tooth angle

$$\psi(r) = \frac{s + \Delta s}{d} + \mathrm{inv}\,\alpha - \mathrm{inv}\,\alpha_r\ \ \text{(external)},\qquad \psi(r) = \frac{s + \Delta s}{d} - \mathrm{inv}\,\alpha + \mathrm{inv}\,\alpha_r\ \ \text{(internal: the tooth widens outward)},$$

for $r$ from the root form radius $r_{Ff} = \max(d_{Ff}/2,\ r_b)$ to the tip radius $r_a$ (internal: from $r_a$ out to $\min(d_{Ff}/2,\ r_f)$). The tip is an arc of $r_a$ between the two flanks.

**Root fillet (external gears).** In rack coordinates ($u$ along the rolling line from the space centre, $y$ radial) the tool reference line is shifted by $x\,m$ and its tip line is the root circle:

$$y_{ref} = r + x\,m,\qquad y_{tip} = y_{ref} - h_{fP}^*\,m = r_f,\qquad y_c = y_{tip} + \rho_{fP},$$

and the centre of the tool tip radius $\rho_{fP}$ (half tool tooth width $\pi m/4 - \Delta s/2$ on the reference line) is at

$$u_c = \frac{\pi m}{4} - \frac{\Delta s}{2} - (y_{ref} - y_c)\tan\alpha - \frac{\rho_{fP}}{\cos\alpha}.$$

As the gear turns by $\varphi$ the rack moves by $r\varphi$ (the reference circle rolls without slip on the tool reference line), so a tool point $(u, y)$ lands in gear coordinates at

$$X = u - r\varphi,\qquad x_g = X\cos\varphi + y\sin\varphi,\qquad y_g = -X\sin\varphi + y\cos\varphi .$$

Sweeping $\varphi$ over $\pm 3\pi/z$ and taking, in each angular bin of the space below the form circle, the smallest radius reached by the tool gives the fillet: the envelope of the tool tip positions. It joins the flanks at the form circle, where the space half angle is $\pi/z - \psi(r_{Ff})$.

In [ ]:
#| export
def _involute_xy(rb: float, r: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    "Polar angle of the involute at radius r (from the start on the base circle)."
    a = np.arccos(np.clip(rb / r, -1, 1))
    return r, np.tan(a) - a


def _rack_fillet(g: GearGeometry, ds: float, n_bins: int = 60, n_phi: int = 1200) -> tuple[np.ndarray, np.ndarray]:
    """
    Root fillet and root of a rack-generated (hobbed) external gear, as polar points (angle from the
    space centre, radius) across the part of the tooth space below the root form circle.
    The fillet is the envelope of the tool tip radius ρ_fP as the rack rolls on the reference circle.
    """
    m, a = g.m, g.alpha
    r = g.d / 2
    rho = g.rho_f * m
    y_ref = r + g.x * m                          # tool reference line (shifted by x·m)
    y_tip = y_ref - g.hf * m                     # tool tip line = gear root circle
    y_c = y_tip + rho
    half_ref = np.pi * m / 4 - ds / 2            # thinner gear tooth ↔ wider tool tooth
    u_c = half_ref - (y_ref - y_c) * np.tan(a) - rho / np.cos(a)
    t = np.linspace(-np.pi / 2, -a, 40)
    pu = np.concatenate([u_c + rho * np.cos(t), np.linspace(0, max(u_c, 0), 8)])
    py = np.concatenate([y_c + rho * np.sin(t), np.full(8, y_tip)])
    pu = np.concatenate([pu, -pu]); py = np.concatenate([py, py])
    phi = np.linspace(-3 * np.pi / g.z, 3 * np.pi / g.z, n_phi)[:, None]
    X = pu[None, :] - r * phi
    xg = X * np.cos(phi) + py[None, :] * np.sin(phi)
    yg = -X * np.sin(phi) + py[None, :] * np.cos(phi)
    th = (np.arctan2(yg, xg) - np.pi / 2).ravel()
    rr = np.hypot(xg, yg).ravel()
    r_form = max(g.d_form / 2, g.db / 2)
    rb = g.db / 2
    half_form = (g.s + ds) / g.d + float(inv(g.alpha)) - float(inv(np.arccos(min(1.0, rb / r_form))))
    lim = np.pi / g.z - half_form                # space half-angle at the form circle
    edges = np.linspace(-lim, lim, n_bins + 1)
    idx = np.digitize(th, edges) - 1
    ok = (idx >= 0) & (idx < n_bins)
    rmin = np.full(n_bins, np.inf)
    np.minimum.at(rmin, idx[ok], rr[ok])
    centres = (edges[:-1] + edges[1:]) / 2
    good = np.isfinite(rmin)
    rmin = np.interp(centres, centres[good], rmin[good])
    rmin = np.minimum(rmin, r_form)
    ang = np.concatenate([[-lim], centres, [lim]])
    rad = np.concatenate([[r_form], rmin, [r_form]])
    return ang, rad


def gear_outline(g: GearGeometry, ds: float | None = None, n_flank: int = 24, n_arc: int = 6) -> np.ndarray:
    """
    Closed outline (N×2 array, mm) of gear g, centred on the origin, one tooth on the +x axis.

    External gears: exact involute flanks from the root form circle to the tip, and the root fillet
    generated by the ISO 53 rack (tool tip radius ρ_fP) below it. Internal gears: involute flanks and
    a straight run from the form circle to the root circle (the real fillet depends on the shaper cutter).
    ds: thickness change on the reference circle [mm] (negative: thinner; default 0).
    """
    ds = 0.0 if ds is None else ds
    s = g.s + ds
    rb = g.db / 2
    ra, rf = g.da / 2, g.df / 2
    r_form = max(g.d_form / 2, rb) if not g.internal else min(g.d_form / 2, rf)
    psi_ref = s / g.d                                  # half tooth angle on the reference circle
    inv_a = float(inv(g.alpha))
    pitch = 2 * np.pi / g.z
    if g.internal:
        r = np.linspace(ra, r_form, n_flank)           # from tip (inside) outward
        _, ang = _involute_xy(rb, np.maximum(r, rb))
        half = psi_ref - inv_a + ang                   # internal tooth widens outward
    else:
        r = np.linspace(r_form, ra, n_flank)
        _, ang = _involute_xy(rb, r)
        half = psi_ref + inv_a - ang
        f_ang, f_rad = _rack_fillet(g, ds)
    pts = []
    for k in range(g.z):
        c = k * pitch
        if g.internal:
            # r runs tip (inner) → form (outer). Right flank root→tip, tip arc across the tooth,
            # left flank tip→root, root arc across the space to the next tooth.
            right = [(rf, c - half[-1])] + [(ri, c - h) for ri, h in zip(r[::-1], half[::-1])]
            tip = [(ra, t) for t in np.linspace(c - half[0], c + half[0], n_arc)[1:-1]]
            left = [(ri, c + h) for ri, h in zip(r, half)] + [(rf, c + half[-1])]
            nxt = (k + 1) * pitch - half[-1]
            seq = right + tip + left + [(rf, t) for t in np.linspace(c + half[-1], nxt, n_arc)[1:-1]]
        else:
            # right flank form→tip, tip arc, left flank tip→form, generated root across the next space
            right = [(ri, c - h) for ri, h in zip(r, half)]
            tip = [(ra, t) for t in np.linspace(c - half[-1], c + half[-1], n_arc)[1:-1]]
            left = [(ri, c + h) for ri, h in zip(r[::-1], half[::-1])]
            root = [(ri, c + pitch / 2 + ti) for ti, ri in zip(f_ang[1:-1], f_rad[1:-1])]
            seq = right + tip + left + root
        pts += seq
    P = np.array([(ri * np.cos(t), ri * np.sin(t)) for ri, t in pts])
    return P


def write_dxf(path: str, outlines: list[np.ndarray], circles: tuple | list = ()) -> None:
    "Write closed polylines (and circles: (x, y, r)) to an R12 ASCII DXF file, units mm."
    L = ["0", "SECTION", "2", "HEADER", "9", "$INSUNITS", "70", "4", "0", "ENDSEC",
         "0", "SECTION", "2", "ENTITIES"]
    for P in outlines:
        L += ["0", "POLYLINE", "8", "0", "66", "1", "70", "1"]
        for x, y in P:
            L += ["0", "VERTEX", "8", "0", "10", f"{x:.5f}", "20", f"{y:.5f}"]
        L += ["0", "SEQEND"]
    for x, y, r in circles:
        L += ["0", "CIRCLE", "8", "0", "10", f"{x:.5f}", "20", f"{y:.5f}", "40", f"{r:.5f}"]
    L += ["0", "ENDSEC", "0", "EOF"]
    with open(path, "w") as f:
        f.write("\n".join(L) + "\n")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 6))
P = gear_outline(ps.ring, -0.05)
ax.plot(*P.T, lw=0.8, label="ring")
rho = np.pi * ps.planet.z / ps.ring.z                     # ring phase so its teeth mesh with planet 1
P = P @ np.array([[np.cos(rho), np.sin(rho)], [-np.sin(rho), np.cos(rho)]])
ax.lines[0].set_data(*P.T)
for k in range(3):
    th = 2 * np.pi * k / 3
    psi = th * (1 + ps.sun.z / ps.planet.z) + np.pi - np.pi / ps.planet.z   # planet phase that meshes with the sun
    R_ = np.array([[np.cos(psi), np.sin(psi)], [-np.sin(psi), np.cos(psi)]])
    Pp = gear_outline(ps.planet, -0.05) @ R_ + ps.a_w * np.array([np.cos(th), np.sin(th)])
    ax.plot(*Pp.T, lw=0.8, color="C1")
Ps = gear_outline(ps.sun, -0.05)
ax.plot(*Ps.T, lw=0.8, color="C2")
ax.set_aspect("equal"); ax.set_axis_off()
plt.tight_layout()

Checks on the outline: every point lies between the root and tip circles, and the tooth thickness on the reference circle matches $s$:

In [ ]:
for g in (ps.sun, ps.planet, ps.ring):
    P = gear_outline(g)
    rr = np.hypot(P[:, 0], P[:, 1])
    lo, hi = min(g.da, g.df) / 2, max(g.da, g.df) / 2
    assert rr.min() >= lo - 1e-6 and rr.max() <= hi + 1e-6, g.name
    # tooth thickness on the reference circle, interpolated on the first tooth's two flanks
    th = np.arctan2(P[:, 1], P[:, 0])
    flank = (np.abs(th) < np.pi / g.z) & (rr > lo + 1e-6) & (rr < hi - 1e-6)
    sides = []
    for sgn in (-1, 1):
        sel = flank & (np.sign(th) == sgn)
        order = np.argsort(rr[sel])
        sides.append(np.interp(g.d / 2, rr[sel][order], th[sel][order]))
    s_meas = (sides[1] - sides[0]) * g.d / 2
    print(f"{g.name:6s} thickness on d from the outline {s_meas:.4f} mm, nominal s = {g.s:.4f} mm")
    assert abs(s_meas - g.s) < 1e-3, g.name

---

## Tests

In [ ]:
#| hide
import tempfile, os
# ISO 286 spot checks against the printed tables
assert iso286_limits(30, "H7") == (0.021, 0.0) and iso286_limits(30.5, "H7") == (0.025, 0.0)
assert iso286_limits(8, "j5") == (0.004, -0.002) and iso286_limits(20, "k6") == (0.015, 0.002)
assert iso286_limits(50, "h11") == (0.0, -0.16) and iso286_limits(12, "N7") == (-0.005, -0.023)
assert iso286_limits(14, "P9") == (-0.018, -0.061) and iso286_limits(14, "JS9") == (0.0215, -0.0215)
assert iso286_limits(14, "s6") == (0.039, 0.028) and iso286_limits(20, "r6") == (0.041, 0.028)
assert iso286_limits(25, "u6") == (0.061, 0.041) and iso286_limits(70, "s6") == (0.078, 0.053)
# ISO 1328 class steps by √2
t5 = iso1328_tolerances(GearGeometry(z=40, m=2, b=20), 5)
t7 = iso1328_tolerances(GearGeometry(z=40, m=2, b=20), 7)
assert abs(t7["F_pT"] / t5["F_pT"] - 2) < 0.1
# span limits bracket and differ by T_sn·cos α
al = bl["allowances"]["sun"]
k = span_teeth(ps.sun)
dW = span_measurement(ps.sun, ps.sun.s + al.A_sne, k) - span_measurement(ps.sun, ps.sun.s + al.A_sni, k)
assert math.isclose(dW, al.T_sn * math.cos(ps.sun.alpha))
# internal gear: thinner teeth → larger dimension between balls (first value of the row is the upper limit)
row = next(r for r in ring_sheet if r.item.startswith("Dimension between"))
hi_, lo_ = [float(x) for x in row.value.split(" mm")[0].split(" / ")]
assert hi_ > lo_
# DXF writes and closes
with tempfile.TemporaryDirectory() as td:
    pth = os.path.join(td, "sun.dxf")
    write_dxf(pth, [gear_outline(ps.sun)], [(0, 0, 7.0)])
    txt = open(pth).read()
    assert txt.count("VERTEX") == len(gear_outline(ps.sun)) and txt.strip().endswith("EOF")

## References

- ISO 1328-1:2013 — *Cylindrical gears — ISO system of flank tolerance classification — Part 1*.
- ISO/TR 10064-2:1996 — *Code of inspection practice — Radial composite deviations, runout, tooth thickness and backlash*.
- ISO 1340:1976 and DIN 3966-1:1978 — *Information to be given on drawings of involute gears*.
- DIN 3967:1978 — *Backlash, tooth thickness allowances, tooth thickness tolerances*.
- ISO 286-2:2010 — *Tables of standard tolerance classes and limit deviations for holes and shafts*.
- ISO 2639:2002 — *Steels — Determination and verification of the depth of carburized and hardened cases*.